In [ ]:
import json
import os
from transformers import AutoTokenizer

In [ ]:
def chunk_by_tokens_offsets(text, tokenizer, chunk_size=400, overlap=80):
    encoded = tokenizer(
        text,
        add_special_tokens=False,
        return_offsets_mapping=True
    )

    input_ids = encoded["input_ids"]
    offsets = encoded["offset_mapping"]

    chunks = []
    step = chunk_size - overlap

    for i in range(0, len(input_ids), step):
        window_offsets = offsets[i : i + chunk_size]

        if not window_offsets:
            continue

        start_char = window_offsets[0][0]
        end_char = window_offsets[-1][1]

        chunk_text = text[start_char:end_char]
        chunks.append(chunk_text)

        if i + chunk_size >= len(input_ids):
            break

    return chunks

In [ ]:
def process_literature(input_path, output_path, tokenizer=None):
    if tokenizer is None:
        MODEL_NAME = "pritamdeka/S-PubMedBert-MS-MARCO"
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

        if not tokenizer.is_fast:
            raise RuntimeError("Fast tokenizer required")

    if not os.path.exists(input_path):
        raise FileNotFoundError(f"❌ File not found: {input_path}")

    with open(input_path, "r", encoding="utf-8") as f:
        papers = json.load(f)

    print(f"📚 Loaded papers: {len(papers)}")

    chunked_data = []

    for paper in papers:
        pmid = paper.get("pmid")
        title = paper.get("title", "")
        abstract = paper.get("abstract", "")

        if not abstract:
            continue

        # ✅ Normalize text
        title = title.replace("\n", " ").strip()
        abstract = abstract.replace("\n", " ").strip()

        # ✅ Chunking
        chunks = chunk_by_tokens_offsets(abstract, tokenizer)

        # ✅ Add chunk IDs and FILTER tiny chunks
        for i, chunk in enumerate(chunks):
            # 🛑 The Bouncer: Drop chunks smaller than 40 tokens
            chunk_token_count = len(tokenizer.encode(chunk, add_special_tokens=False))
            if chunk_token_count < 40:
                continue

            chunked_data.append({
                "chunk_id": f"{pmid}_{i}",
                "pmid": pmid,
                "title": title,
                "chunk": chunk
            })

    print(f"🧩 Total chunks created: {len(chunked_data)}")
    
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(chunked_data, f, indent=4, ensure_ascii=False)

    print(f"✅ Saved to {output_path}")

In [ ]:
if __name__ == "__main__":
    INPUT_PATH = "data/literature/SNCA_A53T_corpus.json"
    OUTPUT_PATH = "data/literature/chunked_corpus.json"
    process_literature(INPUT_PATH, OUTPUT_PATH)